# Week 4 Tutorial — Assignment A2, Part A
## Machine Learning: Ergonomic assessment using ML
**In-class activity:** An Ergonomic Assessment Tool With a Smartphone in Pocket

**Part A of A2:** Prepare the dataset, extract and select features, and train a
**Logistic Regression** baseline model.

> **A2 is one assignment with one marked submission, out of 100, worked on across two
> tutorials.** Part A (this notebook, Week 4) is worth **55 marks** and Part B (Week 5)
> the remaining **45** — but they are marked together, once.
>
> **Nothing you submit this week is graded.** Today's upload is a progress record only.
> You answer both parts in the same question document and submit the completed version
> — Parts A and B in one file — at the end of Week 5. That is the submission that is
> marked.

**Data source:** A 20-min video with labelled sensor data of a scaffolder moving steel
bars, including desired amount of standing, walking, bending, and squatting motions.

### Objective

This week you build the **complete machine-learning pipeline end to end** and finish it
with one model: multinomial Logistic Regression. Next week you reuse the same pipeline
to train KNN, SVM and Random Forest, and compare all four.

Doing it in this order is deliberate. A linear model is the simplest thing that can
work, so the score it reaches is the **bar that any more complex model has to clear**.
Without a baseline, a macro F1 of 0.83 from a Random Forest is just a number — you
cannot tell whether the forest earned it or whether the task was easy all along. That
dependency is why the two parts are one assignment: Part B's numbers are meaningless
without Part A's.

Four stages this week:

1. Data preparation
    * Load and check out data info
2. Feature extraction and selection
    * Build window features, then select important features by ranking
3. Model training
    * Train a multinomial Logistic Regression model
4. Model evaluation
    * Assess Accuracy, macro Recall, macro Precision, and macro F1-score
    * Read the confusion matrix and inspect the model coefficients

---
### Steps
1. Setup environment
2. Load preprocessed data
3. Feature extraction and selection
4. Prepare modelling data
5. Logistic Regression baseline
6. Evaluate the baseline on the test set

## Learning outcomes

By the end of this in-class assignment, you should be able to:

1. Inspect an annotated smartphone-sensor dataset and identify the posture labels.
2. Prepare features and labels for a supervised multi-class classification task.
3. Apply feature selection and explain why selected sensor features may help distinguish
   postures.
4. Build a multinomial Logistic Regression classifier inside a scaling pipeline.
5. Use a validation set to support model selection and reserve a test set for final
   evaluation.
6. Evaluate a model using Accuracy, macro Precision, macro Recall, macro F1-score, and a
   confusion matrix.
7. Explain why a simple baseline is measured before more flexible models are tried.

> **Important:** A2 is an **in-class assignment**. Complete this notebook, ensure all
> required outputs are visible, export it as a PDF, and upload it before leaving. That
> upload is a **progress record and is not graded** — the marked submission is the
> combined Part A + Part B document at the end of Week 5.

## 60-minute flow — Part A, questions 1 to 6

| Time | Activity | Student output | Q | Marks |
|---|---|---|---:|---:|
| 0-5 min | Objectives, context, and assessment briefing | Identify the four posture classes and state the multi-class classification objective | 1 | 3 |
| 5-15 min | Inspect data and labels | Dataset shape, variable/data-type check, label identification, and class-distribution table or plot | 2 | 8 |
| 15-30 min | Feature extraction and selection | Top-20 feature list and explanations for at least three selected features | 3 | 12 |
| 30-40 min | Prepare modelling data | Feature matrix and target definition; stratified 60/20/20 train-validation-test split; short note on avoiding data leakage | 4 | 8 |
| 40-52 min | Build the baseline: Logistic Regression | Standardized LR pipeline, validation metrics, validation confusion matrix, and a coefficient reading | 5 | 14 |
| 52-58 min | Baseline test score and interpretation | Test-set metrics recorded as Part B's reference bar | 6 | 10 |
| 58-60 min | Submit Part A progress | Exported PDF with visible code, outputs, figures, and written answers | - | - |
|  | **Part A subtotal** |  |  | **55** |
|  | *Part B (Week 5), questions 7-11* |  |  | *45* |
|  | **Assignment total** |  |  | **100** |

Step 1: Setup environment

In [ ]:
import math
import warnings
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
from google.colab import files
import io

In [ ]:
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
matplotlib.rcParams['font.size'] = 14
plt.rcParams['font.serif'] = 'Times New Roman'
warnings.filterwarnings("ignore")


## Step 2: Inspect the dataset — 15 points
Upload the sensor dataset file downloaded from Canvas assignment 2.
The provided dataset contains smartphone sensor data collected while a construction worker performed four postures:

| Original label | Posture |
|---:|---|
| 1 | Standing |
| 2 | Walking |
| 5 | Bending |
| 8 | Squatting |

### Your task

1. Display the dataset shape, column names, and data types.
2. Identify the label column and the sensor-feature columns.
3. Produce a class-distribution table or bar chart for `Label_2`.
4. In 1–2 sentences, comment on whether the four posture classes appear balanced.

The magnetometer variables are not used because steel reinforcement in the work environment may strongly disturb magnetic signals. Identifier, timestamp, and other non-sensor columns should not be used as model inputs.

In [ ]:
import pandas as pd

github_csv_url = "https://raw.githubusercontent.com/yantaolab/Autonomous-Construction-and-Robotics/main/Projects/Project%2002%20Smartphone-based%20pose%20classification/preprocessed%20dataset%20823.csv"

try:
    df = pd.read_csv(github_csv_url)
except Exception as e:
    try:
        df = pd.read_csv(github_csv_url, sep='\t')
    except Exception:
        raise RuntimeError(f"Failed to load dataset from github url! error:{e}")
print(f"Loaded dataset from GitHub.")

In [ ]:
smoothed_data = df
smoothed_data



1.   Check data frame info.
2.   In the datasert, label_2 (posture) is defined as：
  1.standing
  2.walking
  5.bending
  8.squatting
3.   After checking data, magnetometer is found hugely disturbed by massive amount of steel bars in environment. So Magnetometer_x, Magnetometer_y, Magnetometer_z need to be dropped. Subject names and positions are also irrelevent for further training. Drop them as well.









In [ ]:
smoothed_data = smoothed_data[smoothed_data['Label_2'].isin([1,2,5,8])]

In [ ]:
smoothed_data.columns

In [ ]:
smoothed_data.info()

In [ ]:
label_names_original = {
    1: "Standing",
    2: "Walking",
    5: "Bending",
    8: "Squatting"
}

class_counts = (
    smoothed_data["Label_2"]
    .value_counts()
    .sort_index()
    .rename(index=label_names_original)
)

display(class_counts.to_frame(name="Number of samples"))

plt.figure(figsize=(7, 4))
sns.barplot(
    x=class_counts.index,
    y=class_counts.values,
    hue=class_counts.index,
    palette="Set2",
    legend=False
)
plt.title("Class Distribution of Postures")
plt.xlabel("Posture")
plt.ylabel("Number of samples")
plt.tight_layout()
plt.show()


## Step 3: Feature extraction and selection — 20 points

Smartphone measurements are continuous time-series signals. The feature-extraction code below converts short windows of sensor data into numerical features, including:

- Acceleration and gyroscope magnitude
- Jerk magnitude
- Mean, standard deviation, minimum, maximum, skewness, and kurtosis
- Frequency-domain energy and entropy
- Pitch and roll from gravity measurements
- Correlation between sensor axes

### Your task

After the feature-selection code runs:

1. Report the 20 selected features.
2. Choose at least three selected features from different feature families.
3. For each, explain in 1–2 sentences why the feature may help distinguish the four postures.

Connect your explanation as follows:

**Sensor feature -> body movement or posture -> value for classification**

For example, walking may produce greater acceleration variability and jerk than standing, while pitch and roll may capture body-orientation differences between bending and squatting.

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import skew, kurtosis, entropy
from scipy.fftpack import fft

# Calculate magnitude (combines multi-axis sensor data into a single scalar to represent overall signal strength),
# This step consolidates 3D axis data into a 1D magnitude value, eliminating directionality to focus on the overall intensity of the sensor signal (e.g., total acceleration).

def compute_magnitude(df, cols):
    return np.sqrt((df[cols] ** 2).sum(axis=1))

# Calculate jerk (first derivative of sensor signals, representing the rate of change of motion)
# Jerk reflects how quickly motion changes (e.g., sudden starts/stops), which is a key feature for distinguishing motion patterns (e.g., walking vs. bending).
def compute_jerk(df, cols):
    return df[cols].diff().fillna(0)

# define features statistic
def extract_stat_features(segment):
    return {
        'mean': segment.mean(),
        'std': segment.std(),
        'min': segment.min(),
        'max': segment.max(),
        'skew': skew(segment),
        'kurtosis': kurtosis(segment)
    }

# Extract freqency features
def extract_freq_features(signal):
    fft_vals = np.abs(fft(signal.values))[:len(signal)//2]  #
    energy = np.sum(fft_vals ** 2)
    prob = fft_vals / np.sum(fft_vals)
    spec_entropy = entropy(prob)
    return {
        'fft_energy': energy,
        'fft_entropy': spec_entropy
    }


# Pitch and Roll
def extract_posture_angles(df):
    g = df[['Gravity_x', 'Gravity_y', 'Gravity_z']].values
    pitch = np.arctan2(g[:, 0], np.sqrt(g[:, 1] ** 2 + g[:, 2] ** 2))
    roll = np.arctan2(g[:, 1], g[:, 2])
    return pd.Series({
        'pitch_mean': pitch.mean(),
        'roll_mean': roll.mean(),
        'pitch_std': pitch.std(),
        'roll_std': roll.std()
    })


In [ ]:
def extract_all_features(data, window_size, step):
    features = []
    for start in range(0, len(data) - window_size, step):
        end = start + window_size
        segment = data.iloc[start:end]

        f = {}

        # original magnitude
        f['acc_mag_mean'] = compute_magnitude(segment, ['Accelerometer.xlsx_x','Accelerometer.xlsx_y','Accelerometer.xlsx_z']).mean()
        f['gyro_mag_mean'] = compute_magnitude(segment, ['Gyroscope_x','Gyroscope_y','Gyroscope_z']).mean()

        # jerk and jerk magnitude
        jerk = compute_jerk(segment, ['Accelerometer.xlsx_x','Accelerometer.xlsx_y','Accelerometer.xlsx_z'])
        jerk_mag = compute_magnitude(jerk, ['Accelerometer.xlsx_x','Accelerometer.xlsx_y','Accelerometer.xlsx_z'])
        f['jerk_mag_mean'] = jerk_mag.mean()
        f['jerk_mag_std'] = jerk_mag.std()

        # Extract stat and freq features from accelerometer and gyroscope
        for col in ['Accelerometer.xlsx_x','Accelerometer.xlsx_y','Accelerometer.xlsx_z',
                    'Gyroscope_x','Gyroscope_y','Gyroscope_z']:
            stats = extract_stat_features(segment[col])
            freq = extract_freq_features(segment[col])
            for k, v in stats.items():
                f[f'{col}_{k}'] = v
            for k, v in freq.items():
                f[f'{col}_{k}'] = v

        # Posture angle extraction
        posture_angles = extract_posture_angles(segment)
        f.update(posture_angles)

        # Correspondence calculation
        f['acc_xy_corr'] = segment['Accelerometer.xlsx_x'].corr(segment['Accelerometer.xlsx_y'])
        f['gyro_xy_corr'] = segment['Gyroscope_x'].corr(segment['Gyroscope_y'])

        # Majority vote
        f['label'] = segment['Label_2'].mode()[0]

        features.append(f)
    return pd.DataFrame(features)


In [ ]:
feature_df = extract_all_features(smoothed_data, window_size=5, step=10)

In [ ]:
feature_df

In [ ]:
# label reflection
label_mapping = {1: 0, 2: 1, 5: 2, 8: 3}
feature_df['label'] = feature_df['label'].map(label_mapping)

In [ ]:
# Define all engineered features and the target label
X_all = feature_df.drop(columns=["label"])
y_all = feature_df["label"]

print("Feature matrix shape:", X_all.shape)
print("Target shape:", y_all.shape)


## Step 4: Prepare modelling data — 15 points

Split the data into training, validation, and test sets using a 60/20/20 ratio and stratified sampling.

### Your task

1. Show the number of samples in each split.
2. Explain why stratified splitting is used.
3. Explain the role of each dataset split:
   - Training set: fit the model
   - Validation set: compare settings/models during development
   - Test set: final evaluation only
4. Explain why feature selection, scaling, and any resampling method must be fitted using the training data only.

In [ ]:
from sklearn.model_selection import train_test_split

# First split: 80% development data and 20% final test data
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X_all,
    y_all,
    test_size=0.20,
    random_state=2,
    stratify=y_all
)

# Second split: 75% train and 25% validation within development data
# Overall ratio: 60% train, 20% validation, 20% test
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.25,
    random_state=2,
    stratify=y_train_val
)

print(f"Train set:      {X_train.shape[0]} samples")
print(f"Validation set: {X_val.shape[0]} samples")
print(f"Test set:       {X_test.shape[0]} samples")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE

# Fit feature selection using training data only.
rfe_estimator = RandomForestClassifier(
    n_estimators=100,
    random_state=2,
    n_jobs=-1
)

rfe = RFE(
    estimator=rfe_estimator,
    n_features_to_select=20,
    step=1
)

rfe.fit(X_train, y_train)

selected_features = X_train.columns[rfe.support_].tolist()

feature_ranking = pd.DataFrame({
    "Feature": X_train.columns,
    "Rank": rfe.ranking_,
    "Selected": rfe.support_
}).sort_values(["Selected", "Rank"], ascending=[False, True])

print("Top 20 selected features:")
print(selected_features)
display(feature_ranking.head(20))

In [ ]:
# Retain the same selected features in every dataset split
train_X = X_train[selected_features].copy()
val_X = X_val[selected_features].copy()
test_X = X_test[selected_features].copy()

train_Y = y_train.copy()
val_Y = y_val.copy()
test_Y = y_test.copy()

print("Selected-feature training shape:", train_X.shape)
print("Selected-feature validation shape:", val_X.shape)
print("Selected-feature test shape:", test_X.shape)

## Define evaluation metrics and related functions for following work

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

def evaluate_model(y_true, y_pred, model_name, dataset_name):
    metrics = {
        "Model": model_name,
        "Dataset": dataset_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision (macro)": precision_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "Recall (macro)": recall_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "F1-score (macro)": f1_score(
            y_true, y_pred, average="macro", zero_division=0
        )
    }

    print(f"===== {model_name}: {dataset_name} set =====")
    for metric, value in list(metrics.items())[2:]:
        print(f"{metric}: {value:.4f}")

    print("\nClassification report:")
    print(classification_report(y_true, y_pred, zero_division=0))

    ConfusionMatrixDisplay(
        confusion_matrix=confusion_matrix(y_true, y_pred),
        display_labels=["Standing", "Walking", "Bending", "Squatting"]
    ).plot(cmap="Blues", values_format="d")

    plt.title(f"{model_name}: {dataset_name} confusion matrix")
    plt.tight_layout()
    plt.show()

    return metrics

## Step 5: Logistic Regression baseline — Question 5, 14 points

Build a multinomial Logistic Regression classifier using the 20 selected features.

Logistic Regression draws a **straight boundary** between classes in feature space. For
four postures it fits four sets of weights — one per posture — and predicts whichever
posture gets the highest weighted score. That is the whole model: no neighbours, no
kernel, no trees.

Two settings are worth understanding rather than just copying:

- **`StandardScaler` in the pipeline.** The features are on wildly different scales —
  `fft_energy` runs into the hundreds while `pitch_mean` is a radian value under 3.2.
  Logistic Regression penalises large coefficients (that is what `C` controls), so
  without scaling the penalty would fall almost entirely on the small-scale features
  regardless of how informative they are. Scaling puts every feature on equal footing
  *before* the penalty is applied.
- **`class_weight="balanced"`.** Bending is about 6% of the data. Left alone, the
  optimiser can lower its overall loss by mostly ignoring bending. `balanced` weights
  each class inversely to its frequency, so a bending mistake costs more than a
  squatting mistake.

### Your task

1. Run the pipeline below and train it on the training set only.
2. Evaluate it on the validation set.
3. Report Accuracy, macro Precision, macro Recall, macro F1-score, and the confusion
   matrix.
4. In 1-2 sentences, identify which posture is classified best and which worst, using
   the confusion matrix as evidence.
5. Inspect the coefficient table produced in the next cell and name the feature with the
   largest positive weight for **bending**. In 1-2 sentences, say whether that makes
   physical sense.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

logreg_model = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(
        max_iter=2000,
        C=1.0,
        class_weight="balanced",
        solver="lbfgs",
        random_state=42,
        n_jobs=-1
    ))
])

# Train on the training set only
logreg_model.fit(train_X, train_Y)

# Predictions
logreg_val_pred = logreg_model.predict(val_X)

# Evaluation on the validation set
logreg_val_metrics = evaluate_model(
    val_Y, logreg_val_pred, "Logistic Regression", "Validation"
)

### Reading the coefficients

Logistic Regression is one of the few models that will tell you directly what it
learned. Each class has one weight per feature. Because the features were standardized,
the weights are comparable to each other: a large positive weight means "a high value of
this feature pushes the prediction towards this posture".

This is not available from KNN or SVM with an RBF kernel, and it is a large part of why a
linear baseline is worth fitting even when you expect something else to win.

In [ ]:
# One row per selected feature, one column per posture
coef_df = pd.DataFrame(
    logreg_model.named_steps["logreg"].coef_.T,
    index=selected_features,
    columns=["Standing", "Walking", "Bending", "Squatting"]
).round(3)

display(coef_df)

# The five features that push hardest towards each posture
for posture in coef_df.columns:
    top = coef_df[posture].sort_values(ascending=False).head(5)
    print(f"\nStrongest positive weights for {posture}:")
    for feature, weight in top.items():
        print(f"  {weight:+.3f}  {feature}")

## Step 6: Baseline test score — Question 6, 10 points

Evaluate the trained baseline **once** on the untouched test set, and record the result.

Next week you will train KNN, SVM and Random Forest on exactly the same features and the
same split, and compare them against this number. That comparison is only meaningful if
you do not touch the test set again between now and then.

### Your task

1. Run the cell below to obtain the test-set metrics.
2. Copy the four test metrics into your answer sheet. **You will need them next week.**
3. In 2-3 sentences, answer: is a macro F1-score around this level good enough to deploy
   an ergonomic-risk tool on a real site? Refer to the per-class results for bending, not
   only to the overall score.

In [ ]:
# The test set is used once, at the end, and not revisited
logreg_test_pred = logreg_model.predict(test_X)

logreg_test_metrics = evaluate_model(
    test_Y, logreg_test_pred, "Logistic Regression", "Test"
)

baseline_summary = pd.DataFrame([logreg_test_metrics]).drop(columns="Dataset")
display(baseline_summary.round(4))

print("\nRecord these four numbers. They are the baseline that next week's")
print("KNN, SVM and Random Forest models have to beat.")

---

## Before you submit

Check that every item below is visible in your exported PDF:

- [ ] Dataset shape, column list and data types.
- [ ] Class-distribution table or bar chart for the four postures.
- [ ] The three split sizes (train / validation / test).
- [ ] The 20 selected features.
- [ ] Logistic Regression validation metrics and confusion matrix.
- [ ] The coefficient table and the per-posture top-5 weights.
- [ ] Logistic Regression **test** metrics.
- [ ] All written answers, including three explained features from three different
      families.

**Filename for this week's upload:** `A2_PartA_<YourName>.pdf`

> **This week's upload is not graded.** It is a progress record only — it carries no
> marks, and nothing you upload today is marked at all. The single marked submission is
> the **complete** document, Parts A and B in one file, at the end of Week 5.
>
> So keep working in the same question document. If you start a fresh one next week, the
> 55 marks for Part A are gone.

## Next week — Part B

Same pipeline, same split, same 20 features — three more models (KNN, SVM, Random
Forest), and a four-way comparison against the baseline you just measured. **Keep your
recorded test metrics**; Part B's first question asks you to reproduce them.